# DR-NRT: Diabetic Retinopathy Detection — Kaggle Runner

This notebook runs the DR-NRT experiment pipeline on Kaggle.

**Setup:**
1. Add the APTOS dataset as a Kaggle dataset at: `meiuwu04/data-split`
2. Upload the `dr-nrt` repo as a Kaggle dataset or use Git clone
3. Set `EXP_ID` below to choose which experiment to run

**Phase G (Ordinal Fine-tuning):**
- G1 (exp 600): CORN loss (random init backbone) — GATE experiment
- Requires `coral-pytorch` (installed in Cell 1)
- Uses offline oversampling (target=1000 per class) — run `scripts/offline_oversample.py` locally or upload pre-generated data

**Kaggle Compatibility:**
- Cell 3 patches `models.py` to skip ImageNet weight loading (Kaggle's torch/torchvision have internal incompatibilities)
- Training from random initialization is still valid: isolates the effect of CORN ordinal loss vs Focal Loss
- For full ImageNet-pretrained G1, run locally or use Colab

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
# DO NOT upgrade torch/torchvision — keep Kaggle's default to avoid CUDA conflicts
!pip install -q albumentations scikit-learn scipy matplotlib opencv-python-headless coral-pytorch

# Verify versions
import torch
import torchvision
print(f"PyTorch: {torch.__version__}, Torchvision: {torchvision.__version__}")

In [ ]:
# ============================================================
# Cell 2: Configuration — Set Experiment ID & Paths
# ============================================================
EXP_ID = 600  # G1: CORN on ImageNet backbone (GATE experiment)
NUM_WORKERS = 2  # Kaggle has 4 cores

# Kaggle dataset paths
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/meiuwu04/data-split/data_split"
KAGGLE_WORKING = "/kaggle/working"

# Oversampled data — upload as a separate Kaggle dataset or generate in-notebook
# Option A: Upload pre-generated oversampled images as dataset "meiuwu04/train-oversampled"
KAGGLE_OVERSAMPLE_DATASET = "/kaggle/input/train-oversampled/train_oversampled"
# Option B: Set to "" to skip offline oversampling (will use raw class distribution)
# KAGGLE_OVERSAMPLE_DATASET = ""

In [ ]:
# ============================================================
# Cell 3: Copy source code to working directory & patch paths
# ============================================================
import shutil
import sys
from pathlib import Path

# If source is in a Kaggle dataset, copy to /kaggle/working so we can write checkpoints
src_input = Path("/kaggle/input/dr-nrt/src")
src_working = Path(KAGGLE_WORKING) / "src"

if src_input.exists() and not src_working.exists():
    shutil.copytree(src_input, src_working)
    # Also copy run_experiment.py if present
    re_input = Path("/kaggle/input/dr-nrt/run_experiment.py")
    if re_input.exists():
        shutil.copy2(re_input, Path(KAGGLE_WORKING) / "run_experiment.py")
    # Copy scripts folder (offline_oversample.py etc.)
    scripts_input = Path("/kaggle/input/dr-nrt/scripts")
    scripts_working = Path(KAGGLE_WORKING) / "scripts"
    if scripts_input.exists() and not scripts_working.exists():
        shutil.copytree(scripts_input, scripts_working)

sys.path.insert(0, KAGGLE_WORKING)

# --- Patch config paths to point at Kaggle data ---
import src.config as config_mod

kaggle_data = Path(KAGGLE_DATA_ROOT)
config_mod.DATA_DIR = kaggle_data
config_mod.TRAIN_CSV = kaggle_data / "train_label.csv"
config_mod.TEST_CSV = kaggle_data / "test_label.csv"
config_mod.TRAIN_IMG_DIR = kaggle_data / "train_split"
config_mod.TEST_IMG_DIR = kaggle_data / "test_split"
config_mod.ROOT_DIR = Path(KAGGLE_WORKING)
config_mod.CHECKPOINT_DIR = Path(KAGGLE_WORKING) / "checkpoints"
config_mod.RESULTS_DIR = Path(KAGGLE_WORKING) / "results"

# Patch dataset module (it imports at module level)
import src.dataset as dataset_mod
dataset_mod.TRAIN_CSV = config_mod.TRAIN_CSV
dataset_mod.TEST_CSV = config_mod.TEST_CSV
dataset_mod.TRAIN_IMG_DIR = config_mod.TRAIN_IMG_DIR
dataset_mod.TEST_IMG_DIR = config_mod.TEST_IMG_DIR

# Patch pseudo_label module
import src.pseudo_label as pseudo_mod
pseudo_mod.TEST_IMG_DIR = config_mod.TEST_IMG_DIR

# --- Patch oversample directory for Phase G experiments ---
oversample_dir = Path(KAGGLE_OVERSAMPLE_DATASET) if KAGGLE_OVERSAMPLE_DATASET else None
if oversample_dir and oversample_dir.exists():
    print(f"Oversampled data found at: {oversample_dir}")
    n_files = len(list(oversample_dir.glob("*.png")))
    print(f"  {n_files} oversampled images available")
else:
    oversample_dir = None
    print("No oversampled data — experiments will use raw class distribution")

# --- Monkey-patch models.py to skip ImageNet weights (Kaggle compatibility) ---
# Kaggle's torch/torchvision have internal incompatibilities that prevent ImageNet weight loading
import src.models as models_mod

_original_build_resnet50 = models_mod._build_resnet50

def _patched_build_resnet50(cfg):
    """ResNet50 builder without ImageNet weights (Kaggle compatibility)."""
    import torch.nn as nn
    import torchvision.models as tvm
    
    # Build model from scratch (weights=None) — avoids Kaggle torch internals issues
    model = tvm.resnet50(weights=None)
    print("⚠ Training from scratch (no ImageNet weights) due to Kaggle environment limitations")
    print("   (This is still a valid experiment: tests CORN vs Focal with random init)")
    
    # Apply GeM pooling if needed
    if cfg.use_gem:
        model.avgpool = models_mod.GeM(p=cfg.gem_p)
    
    # Build classifier head
    in_features = model.fc.in_features
    actual_outputs = cfg.num_outputs
    if cfg.loss_type in ("corn", "cumlink"):
        actual_outputs = cfg.num_outputs - 1
    
    if cfg.head_dropout > 0.0:
        model.fc = nn.Sequential(
            nn.Dropout(p=cfg.head_dropout),
            nn.Linear(in_features, actual_outputs),
        )
    else:
        model.fc = nn.Linear(in_features, actual_outputs)
    
    return model

models_mod._build_resnet50 = _patched_build_resnet50
print("✓ Applied Kaggle compatibility patch (skip ImageNet weights)")

print("\nPaths patched for Kaggle:")
print(f"  TRAIN_CSV:     {config_mod.TRAIN_CSV}")
print(f"  TEST_CSV:      {config_mod.TEST_CSV}")
print(f"  TRAIN_IMG_DIR: {config_mod.TRAIN_IMG_DIR}")
print(f"  TEST_IMG_DIR:  {config_mod.TEST_IMG_DIR}")
print(f"  CHECKPOINT_DIR: {config_mod.CHECKPOINT_DIR}")
print(f"  RESULTS_DIR:   {config_mod.RESULTS_DIR}")

In [ ]:
# ============================================================
# Cell 4: Import pipeline modules
# ============================================================
import logging

import torch
import torchvision
from torch.utils.data import DataLoader

from src.config import get_config
from src.dataset import build_datasets, ContrastiveDRDataset
from src.transforms import get_train_transform, get_val_transform
from src.train import run_training, evaluate_on_test
from src.models import build_model

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("kaggle_runner")

print(f"PyTorch: {torch.__version__}, Torchvision: {torchvision.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# Cell 5: Run Experiment
# ============================================================
cfg = get_config(EXP_ID)

# --- Patch oversample_dir to Kaggle path ---
if oversample_dir and cfg.oversample_target > 0:
    cfg.oversample_dir = str(oversample_dir)
    logger.info(f"Oversampling: target={cfg.oversample_target}, dir={cfg.oversample_dir}")
elif cfg.oversample_target > 0:
    # No oversampled data available — disable oversampling
    logger.warning("Oversampling requested but no data available. Disabling oversample_target.")
    cfg.oversample_target = 0
    cfg.oversample_dir = ""

# --- Patch checkpoint/results dirs (they use ROOT_DIR which was already patched) ---
cfg.ckpt_dir.mkdir(parents=True, exist_ok=True)
cfg.results_dir.mkdir(parents=True, exist_ok=True)

logger.info(f"Running experiment: {cfg.exp_name}")
logger.info(f"  loss_type={cfg.loss_type}, backbone={cfg.backbone}, epochs={cfg.total_epochs}")
logger.info(f"  head_dropout={cfg.head_dropout}, scheduler={cfg.scheduler}, wd={cfg.weight_decay}")

transform_train = get_train_transform(cfg.aug_level)
transform_val = get_val_transform()

train_ds, val_ds, test_ds = build_datasets(cfg, transform_train, transform_val)
logger.info(f"Dataset sizes — Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# --- Build loaders ---
sampler = None
shuffle_train = True
if cfg.use_weighted_sampler:
    from torch.utils.data import WeightedRandomSampler
    targets = [s[1] for s in train_ds.samples]
    class_counts = [0] * 5
    for t in targets:
        class_counts[t] += 1
    weights = [1.0 / class_counts[t] for t in targets]
    sampler = WeightedRandomSampler(weights, num_samples=len(targets), replacement=True)
    shuffle_train = False

train_ds_for_loader = train_ds
if cfg.use_joint_contrastive:
    train_ds_for_loader = ContrastiveDRDataset(train_ds, transform_train)

train_loader = DataLoader(
    train_ds_for_loader, batch_size=cfg.batch_size, shuffle=shuffle_train,
    sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# --- Load pre-trained backbone if specified ---
pretrained_backbone_sd = None
if cfg.load_backbone:
    # For Kaggle: remap local path to Kaggle input dataset
    backbone_path = Path(cfg.load_backbone)
    if not backbone_path.exists():
        # Try under Kaggle input
        kaggle_backbone = Path("/kaggle/input/dr-nrt") / backbone_path.relative_to(Path(cfg.load_backbone).parts[0])
        if kaggle_backbone.exists():
            backbone_path = kaggle_backbone
        else:
            logger.warning(f"Backbone not found: {cfg.load_backbone} — starting from ImageNet")
            backbone_path = None
    if backbone_path and backbone_path.exists():
        pretrained_backbone_sd = torch.load(str(backbone_path), weights_only=True)
        logger.info(f"Loaded backbone from {backbone_path}")

# --- Contrastive pre-training (if applicable) ---
if cfg.use_contrastive_pretrain:
    from src.train import run_contrastive_pretraining
    contrastive_ds = ContrastiveDRDataset(train_ds, transform_train)
    contrastive_loader = DataLoader(
        contrastive_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    pretrained_backbone_sd = run_contrastive_pretraining(cfg, contrastive_loader, device)
    del contrastive_loader, contrastive_ds
    if device.type == "cuda":
        torch.cuda.empty_cache()

# --- Train ---
model = run_training(cfg, train_loader, val_loader, device, pretrained_backbone_sd)

# --- Evaluate ---
metrics = evaluate_on_test(
    model, test_ds, test_loader, cfg, device,
    val_loader=val_loader if cfg.use_optimized_thresholds else None,
)

print(f"\n{'='*60}")
print(f"Experiment {cfg.exp_name} — Final Test Results")
print(f"{'='*60}")
for k, v in metrics.items():
    print(f"  {k:20s}: {v:.4f}")

In [ ]:
# ============================================================
# Cell 6: Display Results — Confusion Matrix & Training Curves
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

results_dir = config_mod.RESULTS_DIR / cfg.exp_name
print(f"Results saved to: {results_dir}")

# Show confusion matrix
cm_path = results_dir / f"{cfg.exp_name}_cm.png"
if cm_path.exists():
    print("\nConfusion Matrix:")
    display(IPImage(filename=str(cm_path)))

# Show training curves
curves_path = results_dir / f"{cfg.exp_name}_curves.png"
if curves_path.exists():
    print("\nTraining Curves:")
    display(IPImage(filename=str(curves_path)))

# Show classification report
report_path = results_dir / f"{cfg.exp_name}_cls_report.txt"
if report_path.exists():
    print("\nClassification Report:")
    print(report_path.read_text())

# List all result files
if results_dir.exists():
    print("\nAll output files:")
    for f in sorted(results_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")